# Tugas Praktikum Pengolahan Citra Digital
## Implementasi Teknik Histogram dan Konvolusi

---

### **Soal 1: Perhitungan Manual Histogram Stretching**

**Deskripsi Soal:**
Buat matriks citra *grayscale* $4 \times 4$ dengan rentang nilai sempit (misalnya $80-120$), lakukan *histogram stretching* ke rentang penuh $0-255$ secara manual, dan sajikan langkah-langkah perhitungannya.

#### **1. Matriks Citra Asli ($I$) & Parameter**

Matriks citra *grayscale* ukuran $4 \times 4$ yang digunakan:

$$I = \begin{bmatrix} 
80 & 90 & 110 & 120 \\
90 & 110 & 115 & 90 \\
110 & 115 & 120 & 110 \\
115 & 120 & 80 & 90 
\end{bmatrix}$$

**Parameter Citra Asli:**
- **Nilai Intensitas Minimum ($r_{\min}$)** = $80$
- **Nilai Intensitas Maksimum ($r_{\max}$)** = $120$
- **Rentang Target ($S_{\min} \dots S_{\max}$)** = $0 \dots 255$ (Jumlah tingkat intensitas $L = 256$, sehingga $L-1 = 255$)

---

#### **2. Rumus Linear Histogram Stretching**

Rumus transformasi linier untuk merentangkan intensitas piksel $r$ menjadi $S$ adalah:

$$S = (L - 1) \times \frac{r - r_{\min}}{r_{\max} - r_{\min}}$$

Substitusi parameter ke dalam rumus:

$$S = 255 \times \frac{r - 80}{120 - 80} = 255 \times \frac{r - 80}{40}$$

---

#### **3. Langkah Perhitungan Manual Tiap Piksel ($r$)**

1. **Piksel $r = 80$:**
   $$S = 255 \times \frac{80 - 80}{40} = 255 \times 0 = 0$$

2. **Piksel $r = 90$:**
   $$S = 255 \times \frac{90 - 80}{40} = 255 \times \frac{10}{40} = 255 \times 0.25 = 63.75 \approx 64$$

3. **Piksel $r = 110$:**
   $$S = 255 \times \frac{110 - 80}{40} = 255 \times \frac{30}{40} = 255 \times 0.75 = 191.25 \approx 191$$

4. **Piksel $r = 115$:**
   $$S = 255 \times \frac{115 - 80}{40} = 255 \times \frac{35}{40} = 255 \times 0.875 = 223.125 \approx 223$$

5. **Piksel $r = 120$:**
   $$S = 255 \times \frac{120 - 80}{40} = 255 \times \frac{40}{40} = 255 \times 1 = 255$$

---

#### **4. Matriks Hasil Stretching ($S$)**

**a. Hasil Perhitungan Eksak (Nilai Desimal):**
$$S_{\text{eksak}} = \begin{bmatrix} 
0 & 63.75 & 191.25 & 255 \\
63.75 & 191.25 & 223.125 & 63.75 \\
191.25 & 223.125 & 255 & 191.25 \\
223.125 & 255 & 0 & 63.75 
\end{bmatrix}$$

**b. Hasil Pembulatan (Tipe Data `uint8` / $0-255$):**
$$S_{\text{uint8}} = \begin{bmatrix} 
0 & 64 & 191 & 255 \\
64 & 191 & 223 & 64 \\
191 & 223 & 255 & 191 \\
223 & 255 & 0 & 64 
\end{bmatrix}$$

**Kesimpulan:** Rentang intensitas citra yang semula sempit ($80 \dots 120$) berhasil direntangkan penuh ke rentang $0 \dots 255$ sehingga kontras citra meningkat secara signifikan.

In [ ]:
import numpy as np

# 1. Matriks citra asli 4x4
I = np.array([
    [80, 90, 110, 120],
    [90, 110, 115, 90],
    [110, 115, 120, 110],
    [115, 120, 80, 90]
], dtype=np.uint8)

# 2. Parameter citra
r_min = I.min()
r_max = I.max()
L_minus_1 = 255

# 3. Perhitungan manual linier histogram stretching (sesuai rumus)
S_float = L_minus_1 * (I.astype(float) - r_min) / (r_max - r_min)
S_uint8 = np.round(S_float).astype(np.uint8)

# Display hasil
print("Matriks Asli (I):")
print(I)
print("\nHasil Stretching Eksak (Float):")
print(np.round(S_float, 3))
print("\nHasil Stretching Pembulatan (uint8):")
print(S_uint8)


---

### **Soal 2: Penerapan dan Perbandingan Median Filter vs Mean Filter**

**Deskripsi Soal:**
Terapkan *Median Filter* $3 \times 3$ secara manual pada matriks $5 \times 5$ yang mengandung titik *noise*, lalu bandingkan hasilnya dengan *Mean Filter* pada data yang sama dan jelaskan perbedaan hasilnya.

#### **1. Matriks Citra Input ($5 \times 5$)**

Diberikan matriks citra *grayscale* ukuran $5 \times 5$ dengan latar belakang bernilai sekitar $48-53$ dan disisipi 3 titik *Salt & Pepper Noise*:

$$I_2 = \begin{bmatrix}
50 & 52 & 48 & 51 & 49 \\
51 & \mathbf{0} & 53 & 50 & 52 \\
49 & 50 & \mathbf{255} & 48 & 51 \\
52 & 51 & 49 & \mathbf{0} & 50 \\
50 & 48 & 52 & 51 & 49
\end{bmatrix}$$

**Daftar Titik Noise:**
- **Noise 1 (Pepper)** = $0$ pada posisi $[1, 1]$
- **Noise 2 (Salt)** = $255$ pada posisi $[2, 2]$
- **Noise 3 (Pepper)** = $0$ pada posisi $[3, 3]$

---

#### **2. Langkah Perhitungan Manual pada Piksel Pusat $[2, 2]$**

Sub-matriks jendela $3 \times 3$ di sekitar piksel pusat $I_2[2,2]$ ($r = 255$):

$$\text{Jendela } 3 \times 3 = \begin{bmatrix}
0 & 53 & 50 \\
50 & 255 & 48 \\
51 & 49 & 0
\end{bmatrix}$$

##### **a. Median Filter $3 \times 3$:**
1. Pengurutan (*sorting*) 9 nilai piksel tetangga dari terkecil ke terbesar:
   $$\text{Urutan} = [0, 0, 48, 49, \mathbf{50}, 50, 51, 53, 255]$$
2. Penentuan nilai tengah (elemen ke-5):
   $$\text{Median} = \mathbf{50}$$
   *Nilai piksel $255$ (noise) berhasil dipulihkan menjadi $50$.*

##### **b. Mean Filter $3 \times 3$:**
1. Perhitungan rata-rata 9 piksel tetangga:
   $$\text{Mean} = \frac{0 + 53 + 50 + 50 + 255 + 48 + 51 + 49 + 0}{9} = \frac{556}{9} \approx \mathbf{61.78} \approx \mathbf{62}$$
   *Nilai piksel terdistorsi menjadi $62$ karena terpengaruh nilai noise $255$.*

---

#### **3. Matriks Hasil Filter Bagian Dalam ($3 \times 3$)**

Perbandingan hasil filter pada area internal matriks:

$$\text{Hasil Median Filter} = \begin{bmatrix} 
50 & 50 & 51 \\
51 & \mathbf{50} & 50 \\
50 & 50 & 50
\end{bmatrix}
\quad \text{vs} \quad
\text{Hasil Mean Filter} = \begin{bmatrix} 
68 & 67 & 73 \\
68 & \mathbf{62} & 68 \\
73 & 67 & 67
\end{bmatrix}$$

---

#### **4. Perbandingan & Analisis Hasil**

1. **Median Filter**:
   Sebagai filter statistik non-linier, median filter sangat efektif merestorasi *Salt & Pepper Noise*. Karena menggunakan mekanisme pengurutan (*sorting*), nilai ekstrem ($0$ dan $255$) selalu terdorong ke posisi paling ujung (kiri/kanan). Nilai median yang terpilih berasal dari sinyal citra/latar belakang asli (sekitar $50$), sehingga noise berhasil dihilangkan secara sempurna tanpa memburamkan garis tepi (*edge*).

2. **Mean Filter**:
   Sebagai filter linier, mean filter kurang efektif untuk jenis *Salt & Pepper Noise*. Nilai ekstrem ($0$ dan $255$) ikut terjumlahkan dalam perhitungan rata-rata, sehingga efek noise menyebar ke piksel tetangga yang bersih dan menyebabkan citra menjadi kabur (*blur*).

In [2]:
import numpy as np
from scipy.ndimage import median_filter, uniform_filter

# 1. Matriks citra asli 5x5 dengan 3 titik noise (Pepper=0, Salt=255, Pepper=0)
I2 = np.array([
    [50, 52, 48, 51, 49],
    [51,  0, 53, 50, 52],
    [49, 50, 255, 48, 51],
    [52, 51, 49,  0, 50],
    [50, 48, 52, 51, 49]
], dtype=np.uint8)

# 2. Median Filter 3x3
median_res = median_filter(I2, size=3, mode='nearest')

# 3. Mean Filter 3x3
mean_res = np.round(uniform_filter(I2.astype(float), size=3, mode='nearest')).astype(np.uint8)

print("Matriks Citra Asli (5x5):")
print(I2)

print("\nHasil Median Filter 3x3 (Bagian Dalam 3x3):")
print(median_res[1:4, 1:4])

print("\nHasil Mean Filter 3x3 (Bagian Dalam 3x3):")
print(mean_res[1:4, 1:4])


Matriks Citra Asli (5x5):
[[ 50  52  48  51  49]
 [ 51   0  53  50  52]
 [ 49  50 255  48  51]
 [ 52  51  49   0  50]
 [ 50  48  52  51  49]]

Hasil Median Filter 3x3 (Bagian Dalam 3x3):
[[50 50 51]
 [51 50 50]
 [50 50 50]]

Hasil Mean Filter 3x3 (Bagian Dalam 3x3):
[[68 67 73]
 [68 62 68]
 [73 67 67]]
